In [1]:
from importlib.metadata import version
print(version("torch"))

2.8.0+cu129


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass


In [2]:
@dataclass
class ModelArgs:
    n_heads:int
    dim:int
    hidden_dim:int
    dropout:float
    max_seq_len:int
    n_layer:int

实现多头注意力机制

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, args:ModelArgs,is_casual=False):
        super().__init__()
        assert args.dim%args.n_heads == 0
        self.heads_dim =args.dim//args.n_heads
        self.dim = args.dim
        self.n_heads = args.n_heads
      #实例化is_casual 可以在forward中直接调用self.is_casual
        self.is_casual = is_casual


        self.wq = nn.Linear(args.dim,args.dim,bias=False)
        self.wk = nn.Linear(args.dim,args.dim,bias=False)
        self.wv = nn.Linear(args.dim,args.dim,bias=False)
        self.wo = nn.Linear(args.dim,args.dim,bias=False)

        self.attn_dropout = nn.Dropout(args.dropout)
        self.res_dropout = nn.Dropout(args.dropout)

        if is_casual:
            mask = torch.full((1,1, args.max_seq_len, args.max_seq_len),float("-inf"))
            mask = torch.triu(mask,diagonal=1)
        #PyTorch 的 .to() 方法只会移动模型内部的“参数（Parameters）”和“缓冲区（Buffers）”。它不会去管普通的 Python 属性。
            self.register_buffer( "mask", mask )

    def forward(self,q,k,v):
        batch_size,seq_len,dim = q.shape
        Q, K, V = self.wq(q), self.wk(k), self.wv(v)

        #拆分成多头
        Q=Q.view(batch_size,seq_len,self.n_heads,self.heads_dim)
        K=K.view(batch_size,seq_len,self.n_heads,self.heads_dim)
        V=V.view(batch_size,seq_len,self.n_heads,self.heads_dim)

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        scores = torch.matmul(Q,K.transpose(2,3))/math.sqrt(self.heads_dim)

        if self.is_casual:
            # 直接相加，-inf叠加的位置就等于mask掉了, max_seq_len可能大于seq_len
            scores = scores + self.mask[:, :, :seq_len, :seq_len]
            #如果加了 return scores 函数会提前结束，所以不建议加

        #float 是为了 FP16 的 scores 临时转换为 FP32（单精度，type 是为了将结果转换回输入张量 Q 的原始类型（通常是 FP16 或 BF16）
        scores = F.softmax(scores.float(), dim=-1).type_as(Q)
        scores = self.attn_dropout(scores)
        output = torch.matmul(scores, V)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        #输出层+残差
        output = self.wo(output)
        output = self.res_dropout(output)

        return output


实现层归一化

In [4]:
class LayerNorm(nn.Module):
    def __init__(self,dim,eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

        self.eps =eps
    def forward(self,x):
        mean = x.mean(-1,keepdim=True)
        std = x.std(-1, keepdim=True)

        return self.gamma*(x-mean)/(std+self.eps) + self.beta

实现前馈神经网络FFN

In [8]:
class FFN(nn.Module):
    def __init__(self, dim:int,hidden_dim:int,dropout:float):
        super().__init__()

        self.w1 = nn.Linear(dim,hidden_dim,bias=False)
        self.w2 = nn.Linear(hidden_dim,dim,bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self,x):
         return self.dropout(self.w2(F.relu(self.w1(x))))

编码器

In [14]:
class EncoderLayer(nn.Module):
    #Encoder层的核心实现
    def __init__(self,args):
        super().__init__()

        #定义多头注意力机制层
        self.attention = MultiHeadAttention(args,is_casual=False)

        #定义FFN层
        self.feed_forward = FFN(args.dim,args.hidden_dim,args.dropout)

        #定义俩个Layer norm，分别在MHA之前和FFN之前
        self.attention_norm = LayerNorm(args.dim)
        self.ffn_norm = LayerNorm(args.dim)

    def forward(self,x):
        #归一化层
        norm_x = self.attention_norm(x)

        #多头注意力机制 + 残差连接
        h = x + self.attention.forward(norm_x,norm_x,norm_x)

        #归一化层
        norm_h = self.ffn_norm(h)

        #前馈神经网络 + 残差连接
        out = h +self.feed_forward(norm_h)

        return out

In [10]:
#构建多层Encoder
class Encoder(nn.Module):
    def __init__(self,args):
        super().__init__()

        #多层block
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layer)])

        #最后输出还会有一个归一化层
        self.norm = LayerNorm(args.dim)

    def forward(self,x):
        #分别拖过n_layer层Encoder Layer
        for layer in self.layers:
            x = layer(x)
            return self.norm(x)

Case1

In [11]:
args = ModelArgs(n_heads=8, dim=768, hidden_dim=768*4, dropout=0.1, max_seq_len=512, n_layer=6)
print(args)

ModelArgs(n_heads=8, dim=768, hidden_dim=3072, dropout=0.1, max_seq_len=512, n_layer=6)


In [15]:
batch_size = 10
# 定义输入
x = torch.randn(batch_size, args.max_seq_len, args.dim)

# Encoder
encoder = Encoder(args)

encoder_output = encoder(x)

print("encoder output shape: ", encoder_output.shape)
print("encoder output: ", encoder_output)

encoder output shape:  torch.Size([10, 512, 768])
encoder output:  tensor([[[-3.3676e-02,  2.3936e-01, -6.3493e-01,  ..., -1.7264e+00,
          -1.7801e-01,  1.1126e+00],
         [-8.6164e-01, -4.3570e-01, -3.4488e-01,  ...,  3.5967e-01,
           4.6715e-01, -6.1278e-01],
         [-1.2850e+00, -2.4538e+00,  4.4176e-01,  ...,  4.4771e-01,
          -1.3930e+00,  6.2662e-02],
         ...,
         [-3.5343e-01, -2.9786e-01, -3.3561e-01,  ..., -6.3894e-01,
          -1.0218e+00,  8.0514e-01],
         [-1.0105e+00, -2.5583e-01, -3.8198e-02,  ...,  4.1058e-01,
           9.8020e-01, -6.9436e-02],
         [ 3.0251e-01, -4.8364e-01,  5.4217e-01,  ...,  1.0908e-01,
           7.9613e-01, -1.1135e+00]],

        [[-6.3159e-02, -2.5799e-01,  4.1123e-01,  ..., -1.1380e+00,
          -7.1787e-02, -1.1132e+00],
         [-3.1515e-01, -5.3485e-01, -4.1941e-01,  ..., -5.0537e-01,
          -1.1183e+00,  2.9991e+00],
         [-3.3568e-01,  2.6529e-01,  5.3196e-01,  ..., -1.3211e+00,
         